In [2]:
from pathlib import Path
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency

DATA_DIR = Path('./type3_sample_csvs')

print('설정된 데이터 폴더:', DATA_DIR)

# 0-2. 파일 존재 여부 확인
required_files = ['channel_purchase.csv', 'gender_preference.csv']

print('[파일 존재 여부 확인]')
for fname in required_files:
    fpath = DATA_DIR / fname
    print(f'{fname}:', 'OK' if fpath.exists() else '없음')

설정된 데이터 폴더: type3_sample_csvs
[파일 존재 여부 확인]
channel_purchase.csv: OK
gender_preference.csv: OK


---
# 실습 1. channel_purchase.csv : 카이제곱 독립성 검정

## 1-1. 문제 제시

한 온라인 쇼핑몰은 고객의 **유입채널(`channel`)** 과 **구매여부(`purchase_yn`)** 사이에 관련성이 있는지 알고 싶어 한다.

유입채널은 다음과 같이 구성되어 있다.
- 검색광고
- SNS
- 이메일
- 직접방문

구매여부는 다음과 같이 구성되어 있다.
- 1: 구매
- 0: 비구매

이때, 유입채널과 구매여부가 서로 독립인지 카이제곱 검정을 통해 확인하시오.

---

## 1-2. 가설 설정

- **귀무가설(H0)**: 유입채널과 구매여부는 서로 독립이다.
- **대립가설(H1)**: 유입채널과 구매여부는 서로 독립이 아니다.

---

## 1-3. 해석 포인트

- `pd.crosstab()` 으로 **교차표**를 만든다.
- `chi2_contingency()` 로 **카이제곱 통계량, p-value, 자유도, 기대도수**를 구한다.
- `p-value < 0.05` 이면 유입채널과 구매여부는 **관련이 있다**고 해석한다.

In [3]:
# 1-4. 데이터 불러오기
file_path = DATA_DIR / 'channel_purchase.csv'
df = pd.read_csv(file_path)

print('[데이터 상위 5행]')
print(df.head())
print('[기본 정보]')
print(df.info())

# 1-5. 교차표 생성
ct = pd.crosstab(df['channel'], df['purchase_yn'])

print('[교차표]')
print(ct)

# 1-6. 카이제곱 독립성 검정
chi2, p, dof, expected = chi2_contingency(ct)
expected_df = pd.DataFrame(expected, index=ct.index, columns=ct.columns)

print('[검정 결과]')
print('chi-square statistic:', round(chi2, 4))
print('p-value:', round(p, 6))
print('degrees of freedom:', dof)

print('[기대도수]')
print(expected_df)

# 1-7. 채널별 구매전환율 확인
conversion_rate = pd.crosstab(df['channel'], df['purchase_yn'], normalize='index') * 100

print('[채널별 구매 비율(%)]')
print(conversion_rate.round(2))

print('[최종 해석 출력]')
if p < 0.05:
    print('p-value가 0.05보다 작으므로, 유입채널과 구매여부는 독립이 아니며 서로 관련이 있다고 해석할 수 있습니다.')
else:
    print('p-value가 0.05 이상이므로, 유입채널과 구매여부의 관련성을 확인하기 어렵습니다.')

best_channel = conversion_rate[1].idxmax() if 1 in conversion_rate.columns else conversion_rate.iloc[:, -1].idxmax()
best_rate = conversion_rate[1].max() if 1 in conversion_rate.columns else conversion_rate.iloc[:, -1].max()
print(f'구매 비율이 가장 높은 채널은 {best_channel}이며, 구매 비율은 {best_rate:.2f}%입니다.')

[데이터 상위 5행]
  channel  purchase_yn
0    검색광고            1
1    검색광고            1
2    검색광고            1
3    검색광고            1
4    검색광고            1
[기본 정보]
<class 'pandas.DataFrame'>
RangeIndex: 530 entries, 0 to 529
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   channel      530 non-null    str  
 1   purchase_yn  530 non-null    int64
dtypes: int64(1), str(1)
memory usage: 8.4 KB
None
[교차표]
purchase_yn   0    1
channel             
SNS          55   70
검색광고         50   90
이메일          80   45
직접방문         30  110
[검정 결과]
chi-square statistic: 51.716
p-value: 0.0
degrees of freedom: 3
[기대도수]
purchase_yn          0          1
channel                          
SNS          50.707547  74.292453
검색광고         56.792453  83.207547
이메일          50.707547  74.292453
직접방문         56.792453  83.207547
[채널별 구매 비율(%)]
purchase_yn      0      1
channel                  
SNS          44.00  56.00
검색광고         35.71  64.29
이메일 

---
# 실습 2. gender_preference.csv : 카이제곱 동질성 검정

## 2-1. 문제 제시

한 교육 플랫폼은 **성별(`gender`)** 에 따라 **콘텐츠 선호(`content_type`) 분포**가 같은지 알고 싶어 한다.

콘텐츠 유형은 다음과 같이 구성되어 있다.
- 데이터분석
- 마케팅
- 디자인
- 프로그래밍

이때, 남성과 여성의 콘텐츠 선호 분포가 같은지 카이제곱 검정을 통해 확인하시오.

---

## 2-2. 가설 설정

- **귀무가설(H0)**: 성별에 따른 콘텐츠 선호 분포는 같다.
- **대립가설(H1)**: 성별에 따른 콘텐츠 선호 분포는 다르다.

---

## 2-3. 해석 포인트

- 동질성 검정은 **집단 간 분포가 같은지**를 확인하는 문제이다.
- 계산 함수는 독립성 검정과 같지만, 해석 문장은 **분포가 동일/상이하다**로 표현한다.
- 성별별 최다 선호 콘텐츠도 함께 확인하면 보고서가 더 풍부해진다.

In [4]:
# 2-4. 데이터 불러오기
file_path = DATA_DIR / 'gender_preference.csv'
df = pd.read_csv(file_path)

print('[데이터 상위 5행]')
print(df.head())
print('[기본 정보]')
print(df.info())

# 2-5. 교차표 생성
ct = pd.crosstab(df['gender'], df['content_type'])

print('[교차표]')
print(ct)

# 2-6. 카이제곱 동질성 검정
chi2, p, dof, expected = chi2_contingency(ct)
expected_df = pd.DataFrame(expected, index=ct.index, columns=ct.columns)

print('[검정 결과]')
print('chi-square statistic:', round(chi2, 4))
print('p-value:', round(p, 6))
print('degrees of freedom:', dof)

print('[기대도수]')
print(expected_df)

# 2-7. 성별 내 비율과 최다 선호 콘텐츠 확인
ratio = pd.crosstab(df['gender'], df['content_type'], normalize='index') * 100
top_pref = ct.idxmax(axis=1)

print('[성별 내 콘텐츠 선호 비율(%)]')
print(ratio.round(2))

print('[성별별 최다 선호 콘텐츠]')
print(top_pref)

print('[최종 해석 출력]')
if p < 0.05:
    print('p-value가 0.05보다 작으므로, 성별에 따라 콘텐츠 선호 분포가 다르다고 해석할 수 있습니다.')
else:
    print('p-value가 0.05 이상이므로, 성별에 따른 콘텐츠 선호 분포 차이를 확인하기 어렵습니다.')

for g in top_pref.index:
    print(f'{g}의 최다 선호 콘텐츠는 {top_pref[g]}입니다.')

[데이터 상위 5행]
  gender content_type
0     남성        데이터분석
1     남성        데이터분석
2     남성        데이터분석
3     남성        데이터분석
4     남성        데이터분석
[기본 정보]
<class 'pandas.DataFrame'>
RangeIndex: 330 entries, 0 to 329
Data columns (total 2 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   gender        330 non-null    str  
 1   content_type  330 non-null    str  
dtypes: str(2)
memory usage: 5.3 KB
None
[교차표]
content_type  데이터분석  디자인  마케팅  프로그래밍
gender                              
남성               50   18   28     54
여성               42   48   55     35
[검정 결과]
chi-square statistic: 24.6478
p-value: 1.8e-05
degrees of freedom: 3
[기대도수]
content_type      데이터분석   디자인        마케팅      프로그래밍
gender                                             
남성            41.818182  30.0  37.727273  40.454545
여성            50.181818  36.0  45.272727  48.545455
[성별 내 콘텐츠 선호 비율(%)]
content_type  데이터분석    디자인    마케팅  프로그래밍
gender                                  
남성